# model_random_forest.py

Random Forest for Heart Disease / Cardiovascular Risk prediction.

Steps:
  1. Load preprocessed data
  2. Define model architecture
  3. Loss: Gini impurity (built-in, cross-entropy variant available)
  4. Optimiser: n_estimators / max_depth (tree ensemble hyperparams)
  5. Train with warm_start batching (simulated epochs) + OOB curve
  6. Evaluate
  7. Fine-tune with RandomizedSearchCV
  8. Save model + feature-importance plot
  9. Predict on new data

In [1]:
pip install numpy pandas joblib scikit-learn matplotlib

Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-2.4.6-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached matplotlib-3.10.9-cp312-cp312-win_amd64.whl.metadata (52 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.0-cp312-cp312-win_amd64.whl.metadata (5.2 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------- ----------------------- 5.0/12.3 MB 25.2 MB/s eta 0:00:01
   -


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\inaug\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import joblib, os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, accuracy_score, ConfusionMatrixDisplay)
from scipy.stats import randint
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────────
PRE  = "preprocessed"
MDIR = "models/rf";   os.makedirs(MDIR, exist_ok=True)
PDIR = "plots/rf";    os.makedirs(PDIR, exist_ok=True)
LABEL = {0: 'Low', 1: 'Moderate', 2: 'High'}

## 1. Load preprocessed splits

In [3]:
X_train = pd.read_csv(f"{PRE}/X_train.csv")
X_val   = pd.read_csv(f"{PRE}/X_val.csv")
X_test  = pd.read_csv(f"{PRE}/X_test.csv")
y_train = pd.read_csv(f"{PRE}/y_train.csv").squeeze()
y_val   = pd.read_csv(f"{PRE}/y_val.csv").squeeze()
y_test  = pd.read_csv(f"{PRE}/y_test.csv").squeeze()
class_weights = joblib.load(f"{PRE}/class_weights.pkl")
feature_names = joblib.load(f"{PRE}/feature_names.pkl")
print(f"Train {X_train.shape} | Val {X_val.shape} | Test {X_test.shape}")

# Combine train+val for final fine-tuned model
X_trainval = pd.concat([X_train, X_val], ignore_index=True)
y_trainval = pd.concat([y_train, y_val], ignore_index=True)

Train (51229, 30) | Val (10978, 30) | Test (10978, 30)


## 2. Define model architecture

In [4]:
STEP_TREES = 20    # trees added per "epoch"
N_EPOCHS   = 15    # total epochs → 300 trees at end

rf = RandomForestClassifier(
    n_estimators  = 0,              # start with 0; incremented via warm_start
    criterion     = 'gini',         # 3. Loss: Gini impurity
    max_depth     = 15,
    min_samples_split = 5,
    min_samples_leaf  = 2,
    max_features  = 'sqrt',
    class_weight  = class_weights,
    oob_score     = True,
    warm_start    = True,           # allows incremental tree addition
    n_jobs        = -1,
    random_state  = 42,
)

## 3+4. Loss = Gini; "Optimiser" = growing ensemble

## 5. Train in epochs (warm_start adds STEP_TREES per epoch)

In [5]:
oob_scores, val_acc_hist = [], []
print("\n── Training Random Forest ──")
for epoch in range(1, N_EPOCHS + 1):
    rf.n_estimators = epoch * STEP_TREES
    rf.fit(X_train, y_train)

    oob  = rf.oob_score_
    vacc = accuracy_score(y_val, rf.predict(X_val))
    oob_scores.append(oob)
    val_acc_hist.append(vacc)
    print(f"  Epoch {epoch:02d}/{N_EPOCHS}  trees={rf.n_estimators}  "
          f"OOB={oob:.4f}  val_acc={vacc:.4f}")


── Training Random Forest ──
  Epoch 01/15  trees=20  OOB=0.9869  val_acc=0.9887
  Epoch 02/15  trees=40  OOB=0.9889  val_acc=0.9887
  Epoch 03/15  trees=60  OOB=0.9894  val_acc=0.9889
  Epoch 04/15  trees=80  OOB=0.9894  val_acc=0.9887
  Epoch 05/15  trees=100  OOB=0.9895  val_acc=0.9890
  Epoch 06/15  trees=120  OOB=0.9896  val_acc=0.9888
  Epoch 07/15  trees=140  OOB=0.9898  val_acc=0.9889
  Epoch 08/15  trees=160  OOB=0.9897  val_acc=0.9891
  Epoch 09/15  trees=180  OOB=0.9898  val_acc=0.9892
  Epoch 10/15  trees=200  OOB=0.9896  val_acc=0.9892
  Epoch 11/15  trees=220  OOB=0.9897  val_acc=0.9891
  Epoch 12/15  trees=240  OOB=0.9897  val_acc=0.9892
  Epoch 13/15  trees=260  OOB=0.9898  val_acc=0.9891
  Epoch 14/15  trees=280  OOB=0.9897  val_acc=0.9891
  Epoch 15/15  trees=300  OOB=0.9897  val_acc=0.9892


## 6. Evaluate on Test Set

In [6]:
print("\n── Evaluation on Test Set ──")
y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)
print(classification_report(y_test, y_pred, target_names=[LABEL[i] for i in sorted(LABEL)]))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_proba, multi_class='ovr'))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


── Evaluation on Test Set ──
              precision    recall  f1-score   support

         Low       0.99      1.00      1.00      7669
    Moderate       0.98      0.99      0.99      3173
        High       0.97      0.26      0.42       136

    accuracy                           0.99     10978
   macro avg       0.98      0.75      0.80     10978
weighted avg       0.99      0.99      0.99     10978

Accuracy : 0.9887957733649116
ROC-AUC  : 0.9250728391322761
Confusion Matrix:
 [[7669    0    0]
 [  22 3150    1]
 [  44   56   36]]


## 7. Fine-Tune with RandomizedSearchCV

In [7]:
print("\n── Fine-Tuning (RandomizedSearchCV) ──")
param_dist = {
    'n_estimators'     : randint(200, 600),
    'max_depth'        : [10, 15, 20, 25, None],
    'min_samples_split': randint(2, 10),
    'min_samples_leaf' : randint(1, 6),
    'max_features'     : ['sqrt', 'log2', 0.3, 0.5],
}
rf_search = RandomForestClassifier(
    criterion='gini', class_weight=class_weights, oob_score=False,
    n_jobs=-1, random_state=42
)
rscv = RandomizedSearchCV(
    rf_search, param_dist,
    n_iter=20, cv=StratifiedKFold(3),
    scoring='roc_auc_ovr', n_jobs=-1, verbose=1, random_state=42
)
rscv.fit(X_trainval, y_trainval)
best_rf = rscv.best_estimator_
print("Best params:", rscv.best_params_)

y_pred_ft  = best_rf.predict(X_test)
y_proba_ft = best_rf.predict_proba(X_test)
print("Fine-tuned Accuracy :", accuracy_score(y_test, y_pred_ft))
print("Fine-tuned ROC-AUC  :", roc_auc_score(y_test, y_proba_ft, multi_class='ovr'))
print(classification_report(y_test, y_pred_ft, target_names=[LABEL[i] for i in sorted(LABEL)]))


── Fine-Tuning (RandomizedSearchCV) ──
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best params: {'max_depth': 20, 'max_features': 'log2', 'min_samples_leaf': 5, 'min_samples_split': 3, 'n_estimators': 543}
Fine-tuned Accuracy : 0.9886135908179996
Fine-tuned ROC-AUC  : 0.9393133835707036
              precision    recall  f1-score   support

         Low       0.99      1.00      1.00      7669
    Moderate       0.98      0.99      0.99      3173
        High       1.00      0.26      0.42       136

    accuracy                           0.99     10978
   macro avg       0.99      0.75      0.80     10978
weighted avg       0.99      0.99      0.99     10978



## 8. Save models + plots

In [8]:
joblib.dump(rf,      f"{MDIR}/rf_base.pkl")
joblib.dump(best_rf, f"{MDIR}/rf_finetuned.pkl")
print(f"✅ Models saved to ./{MDIR}/")

# Plots
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Training curve
axes[0].plot(range(STEP_TREES, (N_EPOCHS+1)*STEP_TREES, STEP_TREES), oob_scores,  label='OOB Score')
axes[0].plot(range(STEP_TREES, (N_EPOCHS+1)*STEP_TREES, STEP_TREES), val_acc_hist, label='Val Acc')
axes[0].set(title='RF Training Curve', xlabel='n_estimators', ylabel='Score'); axes[0].legend()

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_ft)
ConfusionMatrixDisplay(cm, display_labels=[LABEL[i] for i in sorted(LABEL)]).plot(ax=axes[1])
axes[1].set_title('RF Confusion Matrix (fine-tuned)')

# Feature importance (top 20)
importances = pd.Series(best_rf.feature_importances_, index=feature_names).nlargest(20)
importances.sort_values().plot(kind='barh', ax=axes[2])
axes[2].set_title('Top-20 Feature Importances')

plt.tight_layout()
plt.savefig(f"{PDIR}/rf_results.png", dpi=150)
print(f"Plot saved to ./{PDIR}/rf_results.png")

✅ Models saved to ./models/rf/
Plot saved to ./plots/rf/rf_results.png


## 9. Predict on new data

In [9]:
def predict_new(raw_df: pd.DataFrame, model_path: str = f"{MDIR}/rf_finetuned.pkl"):
    """
    Accepts a pre-processed DataFrame (post preprocessing.py transformations).
    Returns (predicted_labels, probabilities).
    """
    scaler        = joblib.load(f"{PRE}/scaler.pkl")
    feature_names = joblib.load(f"{PRE}/feature_names.pkl")
    model         = joblib.load(model_path)

    raw_df = raw_df.reindex(columns=feature_names, fill_value=0)

    numerical_cols = [
        'Height_(cm)', 'Weight_(kg)', 'BMI', 'Alcohol_Consumption',
        'Fruit_Consumption', 'Green_Vegetables_Consumption', 'FriedPotato_Consumption',
        'Checkup_Frequency', 'Lifestyle_Score', 'Healthy_Diet_Score',
        'Smoking_Alcohol', 'Checkup_Exercise', 'Height_to_Weight',
        'Fruit_Vegetables', 'HealthyDiet_Lifestyle', 'Alcohol_FriedPotato',
        'Risk_Composite', 'Alcohol_per_Weight'
    ]
    num_present = [c for c in numerical_cols if c in raw_df.columns]
    raw_df[num_present] = scaler.transform(raw_df[num_present])

    preds  = model.predict(raw_df)
    probas = model.predict_proba(raw_df)
    return [LABEL[p] for p in preds], probas

# Demo
demo_preds = best_rf.predict(X_test.iloc[:5].values)
print("\nDemo predictions:", [LABEL[p] for p in demo_preds],
      "| Actual:", list(y_test.iloc[:5].map(LABEL)))


Demo predictions: ['Low', 'Moderate', 'Low', 'Low', 'Low'] | Actual: ['Low', 'Moderate', 'Low', 'Low', 'Low']
